# Athena Verify — Catch RAG Hallucinations Sentence-by-Sentence

**[GitHub](https://github.com/RahulModugula/athena) | [PyPI](https://pypi.org/project/athena-verify/)**

Athena is an open-source runtime guardrail that verifies every sentence in your LLM's answer against the retrieved context. It catches fabricated claims, out-of-context info, number substitutions, and contradictions — before they reach users.

This notebook runs end-to-end in ~2 minutes on a free Colab GPU (T4).

In [ ]:
# Cell 1: Install athena-verify (with NLI support)
!pip install -q "athena-verify[nli]"

from athena_verify import verify

print("athena-verify installed successfully!")
print("NLI model (DeBERTa-v3, ~1.2GB) will download on first use.")

In [ ]:
# Cell 2: Define test cases — a faithful answer and a hallucinated one

context = [
    "The indemnification cap under this Agreement shall be $2,000,000 per incident "
    "and $5,000,000 in the aggregate. The indemnifying party shall not disclose "
    "any confidential information received during the indemnification process. "
    "Claims must be filed within 90 days of discovery.",
]

question = "What are the indemnification terms?"

# A faithful answer — everything comes from the context
good_answer = (
    "The indemnification cap is $2,000,000 per incident and $5,000,000 in the aggregate. "
    "Claims must be filed within 90 days of discovery."
)

# A hallucinated answer — wrong numbers, fabricated details, contradicts context
bad_answer = (
    "The indemnification cap is $1,000,000 per incident. "
    "The indemnifying party is permitted to disclose confidential information. "
    "Claims must be filed within 30 days. "
    "The agreement also includes a force majeure clause covering natural disasters."
)

print("Test cases ready.")
print(f"\nContext ({len(context[0])} chars):\n  {context[0][:100]}...")
print(f"\nGood answer:\n  {good_answer}")
print(f"\nBad answer:\n  {bad_answer}")

In [ ]:
# Cell 3: Run verification — watch it catch every hallucination

print("=" * 60)
print("VERIFYING FAITHFUL ANSWER")
print("=" * 60)
good_result = verify(question=question, answer=good_answer, context=context)
print(f"Trust score:   {good_result.trust_score:.2f}")
print(f"Passed:        {good_result.verification_passed}")
print(f"Supported:     {good_result.supported}")
print(f"Unsupported:   {good_result.unsupported}")

print("\n" + "=" * 60)
print("VERIFYING HALLUCINATED ANSWER")
print("=" * 60)
bad_result = verify(question=question, answer=bad_answer, context=context)
print(f"Trust score:   {bad_result.trust_score:.2f}")
print(f"Passed:        {bad_result.verification_passed}")
print(f"Supported:     {bad_result.supported}")
print(f"Unsupported:   {bad_result.unsupported}")

print("\n" + "=" * 60)
print("PER-SENTENCE BREAKDOWN")
print("=" * 60)
for i, s in enumerate(bad_result.sentences):
    status = "SUPPORTED" if s.label == "supported" else "UNSUPPORTED"
    print(f"  [{status}] (trust={s.trust_score:.2f}) {s.text}")

print(f"\nCaught {len(bad_result.unsupported)} unsupported claims out of {len(bad_result.sentences)} sentences.")

## What happened?

- The **faithful answer** passed verification — every sentence is grounded in the context.
- The **hallucinated answer** was flagged:
  - "$1,000,000 per incident" — number substitution (context says $2M)
  - "permitted to disclose" — contradicts context (says "shall not disclose")
  - "30 days" — number substitution (context says 90 days)
  - "force majeure clause" — fabricated (not in context at all)

## Next steps

- Enable `use_llm_judge=True` for higher accuracy on numbers and negations
- Integrate with LangChain or LlamaIndex (see the [examples](https://github.com/RahulModugula/athena/tree/main/examples))
- Run the full benchmark: `python benchmarks/run_full_eval.py`